In [1]:
import os
print(os.getcwd())

c:\Users\jakem\OneDrive\MLB_Similarity


In [2]:
%ls

 Volume in drive C is Windows
 Volume Serial Number is 3E2E-AC4E

 Directory of c:\Users\jakem\OneDrive\MLB_Similarity

09/29/2026  09:12 PM    <DIR>          .
10/29/2026  10:11 AM    <DIR>          ..
09/24/2026  04:09 PM    <DIR>          .ipynb_checkpoints
09/29/2026  08:58 PM    <DIR>          .venv
09/29/2026  09:11 PM    <DIR>          __pycache__
09/09/2026  09:15 PM             7,491 build_hitter_dataset.py
09/29/2026  09:06 PM             5,015 build_pitcher_dataset.py
09/09/2026  09:29 PM           632,692 hitter_profiles.csv
10/03/2026  01:39 PM            30,826 MLB_Similarity_Engine.ipynb
09/29/2026  09:07 PM         1,084,085 pitcher_profiles.csv
09/29/2026  09:09 PM             4,083 pitcher_similarity_engine.py
09/29/2026  09:09 PM             4,124 similarity_engine.py
09/29/2026  09:13 PM            13,380 similarity_engine_search.ipynb
               8 File(s)      1,781,696 bytes
               5 Dir(s)  359,068,221,440 bytes free


In [3]:
%pip install pybaseball pandas

  Using cached pybaseball-2.2.7-py3-none-any.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pygithub-2.10.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached pynacl-1.6.2-cp38-abi3-win_amd64.whl.metadata (10 kB)
  Using cached pyjwt-2.15.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached cffi-2.1.1-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
Using cached pybaseball-2.2.7-py3-none-any.whl (426 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached pygithub-2.10.0-py3-none-any.whl (455 kB)
Using cached pyjwt-2.15.1-py3-none-any.whl (33 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 35.3 MB/s  0:00:00
Using cached cffi-2.1.1-cp314-cp314-win_amd64.whl (187 kB)
Using cached pynacl-1.6.2-cp38-abi3-win


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%%writefile build_dataset.py
"""
MLB Player Similarity Engine — Data Builder (Baseball-Reference + Savant version)
"""

import argparse
import re
import sys

import pandas as pd

try:
    from pybaseball import (
        batting_stats_bref,
        statcast_batter_exitvelo_barrels,
        statcast_sprint_speed,
        chadwick_register,
        cache,
    )
except ImportError:
    sys.exit("pybaseball is not installed.\nRun:  pip install pybaseball pandas")

cache.enable()


def fetch_bref_batting(start_year: int, end_year: int, min_pa: int) -> pd.DataFrame:
    frames = []
    for year in range(start_year, end_year + 1):
        print(f"Fetching Baseball-Reference batting stats for {year} ...")
        try:
            df = batting_stats_bref(year)
        except Exception as e:
            print(f"  Warning: failed for {year}: {e}")
            continue
        df["season"] = year
        frames.append(df)

    if not frames:
        sys.exit("Couldn't fetch any Baseball-Reference data — check your internet connection.")

    df = pd.concat(frames, ignore_index=True)

    rename = {
        "Name": "name", "Tm": "team", "Age": "age", "PA": "pa", "AB": "ab",
        "BB": "bb", "SO": "so", "HBP": "hbp", "SF": "sf", "IBB": "ibb",
        "2B": "b2", "3B": "b3", "HR": "hr", "BA": "avg", "SLG": "slg",
        "OPS+": "ops_plus", "SB": "sb", "mlbID": "key_mlbam", "bbrefID": "key_bbref",
    }
    keep = [c for c in rename if c in df.columns]
    df = df[keep + ["season"]].rename(columns=rename)

    df = df[pd.to_numeric(df["pa"], errors="coerce") >= min_pa].copy()

    for col in ["pa", "ab", "bb", "so", "hbp", "sf", "ibb", "b2", "b3", "hr"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["k_pct"] = df["so"] / df["pa"]
    df["bb_pct"] = df["bb"] / df["pa"]
    df["iso"] = pd.to_numeric(df["slg"], errors="coerce") - pd.to_numeric(df["avg"], errors="coerce")

    ubb = df["bb"] - df.get("ibb", 0).fillna(0)
    if "H" in df.columns:
        hits = pd.to_numeric(df["H"], errors="coerce")
    else:
        hits = pd.to_numeric(df["avg"], errors="coerce") * df["ab"]
    singles = hits - df["hr"] - df.get("b2", 0).fillna(0) - df.get("b3", 0).fillna(0)

    denom = df["ab"] + df["bb"] - df.get("ibb", 0).fillna(0) + df.get("sf", 0).fillna(0) + df.get("hbp", 0).fillna(0)
    df["woba"] = (
        0.69 * ubb
        + 0.72 * df.get("hbp", 0).fillna(0)
        + 0.89 * singles
        + 1.27 * df.get("b2", 0).fillna(0)
        + 1.62 * df.get("b3", 0).fillna(0)
        + 2.10 * df["hr"]
    ) / denom

    df["name_key"] = df["name"].astype(str).apply(normalize_name)
    return df


def normalize_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    name = re.sub(r"[^a-zA-Z\s]", "", name)
    return re.sub(r"\s+", " ", name).strip().lower()


def fetch_savant(start_year: int, end_year: int) -> pd.DataFrame:
    frames = []
    for year in range(start_year, end_year + 1):
        print(f"Fetching Savant exit velo / barrels for {year} ...")
        try:
            ev = statcast_batter_exitvelo_barrels(year, minBBE=50)
            spd = statcast_sprint_speed(year, min_opp=10)
        except Exception as e:
            print(f"  Warning: Savant fetch failed for {year}: {e}")
            continue

        ev = ev.rename(columns={
            "player_id": "key_mlbam", "avg_hit_speed": "exit_velocity",
            "brl_percent": "barrel_pct", "ev95percent": "hard_hit_pct",
        })
        spd = spd.rename(columns={"player_id": "key_mlbam", "sprint_speed": "sprint_speed"})

        keep_ev = [c for c in ["key_mlbam", "exit_velocity", "barrel_pct", "hard_hit_pct"] if c in ev.columns]
        keep_spd = [c for c in ["key_mlbam", "sprint_speed"] if c in spd.columns]

        merged = ev[keep_ev].merge(spd[keep_spd], on="key_mlbam", how="outer")
        merged["season"] = year
        frames.append(merged)

    if not frames:
        print("  Warning: no Savant data fetched.")
        return pd.DataFrame(columns=["key_mlbam", "season"])

    return pd.concat(frames, ignore_index=True)


def load_savant_extra(path: str) -> pd.DataFrame:
    print(f"Loading optional Savant custom leaderboard: {path}")
    df = pd.read_csv(path)

    col_map = {
        "name": ["last_name, first_name", "player_name", "Name"],
        "gb_pct": ["gb_percent", "GB%"],
        "fb_pct": ["fb_percent", "FB%"],
        "pull_pct": ["pull_percent", "Pull%"],
        "chase_pct": ["chase_percent", "Chase%"],
        "whiff_pct": ["whiff_percent", "Whiff%"],
        "zone_contact_pct": ["z_contact_percent", "Z-Contact%"],
    }
    out = pd.DataFrame()
    for new_col, candidates in col_map.items():
        norm = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in df.columns}
        found = next((norm[re.sub(r'[^a-z0-9]', '', c.lower())] for c in candidates
                      if re.sub(r'[^a-z0-9]', '', c.lower()) in norm), None)
        if found:
            out[new_col] = df[found]

    if "name" in out.columns:
        out["name_key"] = out["name"].apply(normalize_name)
    return out


def fetch_id_crosswalk() -> pd.DataFrame:
    print("Fetching player ID crosswalk (Chadwick register) ...")
    reg = chadwick_register()
    reg = reg[["key_mlbam", "key_bbref"]].dropna()
    reg["key_mlbam"] = reg["key_mlbam"].astype(int)
    return reg


def build_dataset(start_year: int, end_year: int, min_pa: int, savant_extra_path):
    bref = fetch_bref_batting(start_year, end_year, min_pa)
    savant = fetch_savant(start_year, end_year)

    if "key_mlbam" not in bref.columns:
        crosswalk = fetch_id_crosswalk()
        bref = bref.merge(crosswalk, on="key_bbref", how="left")

    bref["key_mlbam"] = pd.to_numeric(bref["key_mlbam"], errors="coerce")
    savant["key_mlbam"] = pd.to_numeric(savant["key_mlbam"], errors="coerce")

    merged = bref.merge(savant, on=["key_mlbam", "season"], how="left")

    if savant_extra_path:
        extra = load_savant_extra(savant_extra_path)
        if "name_key" in extra.columns:
            merged = merged.merge(extra.drop(columns=["name"], errors="ignore"), on="name_key", how="left")

    return merged


def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    core = [c for c in ["k_pct", "bb_pct", "iso", "woba"] if c in df.columns]
    df = df.dropna(subset=["name"] + core)
    print(f"Cleaning: kept {len(df)}/{before} players after dropping incomplete rows.")
    return df.reset_index(drop=True)


def main():
    parser = argparse.ArgumentParser(description="Build the MLB similarity engine dataset (Bref + Savant)")
    parser.add_argument("--start", type=int, default=2024)
    parser.add_argument("--end", type=int, default=2025)
    parser.add_argument("--min-pa", type=int, default=200)
    parser.add_argument("--savant-extra", type=str, default=None)
    parser.add_argument("--out", type=str, default="player_profiles.csv")
    args, _unknown = parser.parse_known_args()

    df = build_dataset(args.start, args.end, args.min_pa, args.savant_extra)
    df = clean_dataset(df)

    df.to_csv(args.out, index=False)
    print(f"\nDone. Wrote {len(df)} player rows x {len(df.columns)} columns to {args.out}")
    print("Columns:", ", ".join(df.columns))


if __name__ == "__main__":
    main()

Writing build_dataset.py


In [5]:
%run build_dataset.py --start 2015 --end 2026 --min-pa 200

Fetching Baseball-Reference batting stats for 2015 ...
Fetching Baseball-Reference batting stats for 2016 ...
Fetching Baseball-Reference batting stats for 2017 ...
Fetching Baseball-Reference batting stats for 2018 ...
Fetching Baseball-Reference batting stats for 2019 ...
Fetching Baseball-Reference batting stats for 2020 ...
Fetching Baseball-Reference batting stats for 2021 ...
Fetching Baseball-Reference batting stats for 2022 ...
Fetching Baseball-Reference batting stats for 2023 ...
Fetching Baseball-Reference batting stats for 2024 ...
Fetching Baseball-Reference batting stats for 2025 ...
Fetching Baseball-Reference batting stats for 2026 ...
Fetching Savant exit velo / barrels for 2015 ...
Fetching Savant exit velo / barrels for 2016 ...
Fetching Savant exit velo / barrels for 2017 ...
Fetching Savant exit velo / barrels for 2018 ...
Fetching Savant exit velo / barrels for 2019 ...
Fetching Savant exit velo / barrels for 2020 ...
Fetching Savant exit velo / barrels for 2021 .

In [6]:
%ls

 Volume in drive C is Windows
 Volume Serial Number is 3E2E-AC4E

 Directory of c:\Users\jakem\OneDrive\MLB_Similarity

10/29/2026  10:21 AM    <DIR>          .
10/29/2026  10:11 AM    <DIR>          ..
09/24/2026  04:09 PM    <DIR>          .ipynb_checkpoints
09/29/2026  08:58 PM    <DIR>          .venv
09/29/2026  09:11 PM    <DIR>          __pycache__
10/29/2026  10:19 AM             7,491 build_dataset.py
09/09/2026  09:15 PM             7,491 build_hitter_dataset.py
09/29/2026  09:06 PM             5,015 build_pitcher_dataset.py
09/09/2026  09:29 PM           632,692 hitter_profiles.csv
10/03/2026  01:39 PM            30,826 MLB_Similarity_Engine.ipynb
09/29/2026  09:07 PM         1,084,085 pitcher_profiles.csv
09/29/2026  09:09 PM             4,083 pitcher_similarity_engine.py
10/29/2026  10:21 AM           767,633 player_profiles.csv
09/29/2026  09:09 PM             4,124 similarity_engine.py
09/29/2026  09:13 PM            13,380 similarity_engine_search.ipynb
              10 

In [7]:
import pandas as pd

df = pd.read_csv("player_profiles.csv")

print("Total rows:", len(df))
print("\nFirst 15 names:")
print(df["name"].head(15).to_list())

print("\nAny null/NaN names?", df["name"].isna().sum())

print("\nSample of names containing commas (Last, First not converted):")
print(df[df["name"].str.contains(",", na=False)]["name"].head(10).to_list())

print("\nSample of names with weird characters (encoding issues):")
weird = df[df["name"].str.contains(r"[^a-zA-Z\s\.\-']", na=False, regex=True)]
print(weird["name"].head(10).to_list())

print("\nDuplicate names (could indicate multiple seasons per player, expected):")
print(df["name"].value_counts().head(10))

Total rows: 4058

First 15 names:
['Jos\\xc3\\xa9 Abreu', 'Dustin Ackley', 'Nick Ahmed', 'Abraham Almonte', 'Yonder Alonso', 'Jose Altuve', 'Pedro \\xc3\\x81lvarez', 'Alexi Amarista', 'Elvis Andrus', 'Nori Aoki', 'Nolan Arenado', 'Cody Asche', 'Alex Avila', 'Mike Aviles', 'Erick Aybar']

Any null/NaN names? 0

Sample of names containing commas (Last, First not converted):
[]

Sample of names with weird characters (encoding issues):
['Jos\\xc3\\xa9 Abreu', 'Pedro \\xc3\\x81lvarez', 'Jos\\xc3\\xa9 Bautista', 'Carlos Beltr\\xc3\\xa1n', 'Adrian Beltr\\xc3\\xa9', 'Andr\\xc3\\xa9s Blanco', 'Gr\\xc3\\xa9gor Blanco', 'Asdr\\xc3\\xbabal Cabrera', 'Robinson Can\\xc3\\xb3', 'Yoenis C\\xc3\\xa9spedes']

Duplicate names (could indicate multiple seasons per player, expected):
name
Jose Altuve              12
Nolan Arenado            12
George Springer          12
Christian Yelich         12
Eugenio Su\xc3\xa1rez    12
Freddie Freeman          12
Paul Goldschmidt         12
Francisco Lindor         1

In [8]:
import pandas as pd
import codecs

def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    try:
        # Turn the literal \xNN escape sequences back into real bytes,
        # then decode those bytes as UTF-8 to get the actual accented character.
        return codecs.escape_decode(text.encode())[0].decode("utf-8")
    except Exception:
        return text  # leave alone if it wasn't actually broken this way

df = pd.read_csv("player_profiles.csv")
df["name"] = df["name"].apply(fix_mojibake)

# Sanity check
print(df["name"].head(25).to_list())

df.to_csv("player_profiles.csv", index=False)
print("\nFixed and re-saved.")

['José Abreu', 'Dustin Ackley', 'Nick Ahmed', 'Abraham Almonte', 'Yonder Alonso', 'Jose Altuve', 'Pedro Álvarez', 'Alexi Amarista', 'Elvis Andrus', 'Nori Aoki', 'Nolan Arenado', 'Cody Asche', 'Alex Avila', 'Mike Aviles', 'Erick Aybar', 'Clint Barmes', 'Brandon Barnes', 'Tucker Barnhart', 'José Bautista', 'Gordon Beckham', 'Tim Beckham', 'Brandon Belt', 'Carlos Beltrán', 'Adrian Beltré', 'Mookie Betts']

Fixed and re-saved.


In [9]:
import pandas as pd

df = pd.read_csv("player_profiles.csv")
sample = df[df["name"].str.contains("Bautista", na=False)]["name"].iloc[0]

print("Displayed:", sample)
print("repr():", repr(sample))
print("Length:", len(sample))

Displayed: José Bautista
repr(): 'José Bautista'
Length: 13


In [10]:
df.to_csv("player_profiles.csv", index=False, encoding="utf-8-sig")

In [11]:
import pandas as pd

df = pd.read_csv("player_profiles.csv")

statcast_cols = ["exit_velocity", "barrel_pct", "hard_hit_pct", "sprint_speed"]
statcast_cols = [c for c in statcast_cols if c in df.columns]

print("Total rows:", len(df))
print()

for col in statcast_cols:
    missing = df[col].isna().sum()
    pct = missing / len(df) * 100
    print(f"{col}: {missing} missing ({pct:.1f}%)")

# Rows missing ALL Statcast columns (likely a name/ID match failure, not just
# a player who didn't qualify for that particular leaderboard)
missing_all = df[statcast_cols].isna().all(axis=1)
print(f"\nMissing every Statcast column: {missing_all.sum()} rows ({missing_all.sum()/len(df)*100:.1f}%)")

# Spot-check: pull a sample of the fully-missing rows so you can eyeball
# whether they're recognizable stars (real merge failures) or replacement-level
# guys who plausibly just didn't meet Savant's minimum batted-ball-event threshold
print("\nSample of players missing all Statcast data:")
print(df[missing_all][["name", "season", "team", "pa"]].sample(min(15, missing_all.sum())))

Total rows: 4058

exit_velocity: 0 missing (0.0%)
barrel_pct: 0 missing (0.0%)
hard_hit_pct: 0 missing (0.0%)
sprint_speed: 0 missing (0.0%)

Missing every Statcast column: 0 rows (0.0%)

Sample of players missing all Statcast data:


Empty DataFrame
Columns: [name, season, team, pa]
Index: []


In [12]:
print("Unique exit_velocity values:", df["exit_velocity"].nunique())
print("Unique barrel_pct values:", df["barrel_pct"].nunique())
print("Unique sprint_speed values:", df["sprint_speed"].nunique())
print()

# If the merge is healthy, different players/seasons should have different values.
# If it's broadcasting, you'll see very few unique values relative to 4040 rows.
print(df[["name", "season", "exit_velocity", "barrel_pct", "sprint_speed"]].sample(10))

Unique exit_velocity values: 146
Unique barrel_pct values: 218
Unique sprint_speed values: 88

                 name  season  exit_velocity  barrel_pct  sprint_speed
392         Jay Bruce    2016           88.9         8.4          26.8
198         Joe Mauer    2015           89.5         3.7          26.5
829   Marwin González    2017           88.7         6.6          26.7
1140    Rafael Devers    2018           90.9         9.1          27.3
2915   Donovan Solano    2023           90.2         6.1          25.4
415     Chris Coghlan    2016           87.2         4.2          26.3
1160   David Fletcher    2018           84.0         0.4          28.1
1734      Kurt Suzuki    2019           86.0         6.5          25.2
3508   Wyatt Langford    2025           91.4        14.0          28.9
3096    Romy Gonzalez    2024           92.3         8.3          28.9


In [16]:
from similarity_engine import find_similar_seasons
from IPython.display import display

# Change these values to any exact player-season present in player_profiles.csv.
query_name = "José Altuve"
query_season = 2015

similar = find_similar_seasons(df, query_name, query_season, n_neighbors=10)
display(similar[["name", "season", "team", "distance"]].to_string(index=False))

'            name  season         team  distance\n     Jose Altuve    2015      Houston  0.000000\n    Nico Hoerner    2025      Chicago  1.856981\n Whit Merrifield    2017  Kansas City  1.893894\nCharlie Blackmon    2015     Colorado  1.915543\n    Nico Hoerner    2023      Chicago  2.000509\n     José Peraza    2018   Cincinnati  2.200504\n     Jean Segura    2016      Arizona  2.201489\n    Bryson Stott    2023 Philadelphia  2.249618\n    Elvis Andrus    2017        Texas  2.277970\n   Jake McCarthy    2026     Colorado  2.281990\n Whit Merrifield    2018  Kansas City  2.306798'